# Import modules python

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
dossier_path = Path("..") /"data" /"1_raw"/ "corpus_zola" # Chemin vers le dossier contenant les fichiers .txt

donnees = [] # Liste pour stocker les données extraites de chaque fichier

# On boucle sur tous les fichiers .txt du dossier
for fichier in dossier_path.glob("*.txt"):
    with open(fichier, "r", encoding="utf-8") as f:
        contenu = f.read()
        
        # On ajoute les données extraites du fichier à la liste
    donnees.append({
            "nom_fichier": fichier.name, # Nom du fichier
            "texte_brut": contenu, # Contenu brut du fichier
        })
      
         

# Création du tableau de bord (DataFrame)
df = pd.DataFrame(donnees)
print(f"{len(df)} textes chargés avec succès.")

31 textes chargés avec succès.


In [3]:
nb_tokens = df["texte_brut"].apply(lambda x: len(x.split())) # Calcul du nombre de tokens pour chaque texte
df["nb_tokens"] = nb_tokens # Ajout de la colonne "nb_tokens" au DataFrame
df

,nom_fichier,texte_brut,nb_tokens
0,1893_20_Le_docteur_Pascal._clean.txt,Dans la chaleur de l’ardente après-midi de jui...,112713
1,1871_1_La_fortune_des_Rougon._clean.txt,Lorsqu’on sort de Plassans par la porte de Rom...,116685
2,1865_La_confession_de_Claude._clean.txt,"Voici l’hiver: l’air, au matin, devient plus f...",47253
3,1899_1_Fecondite._clean.txt,"Ce matin-là, dans le petit pavillon à la lisiè...",220131
4,1887_15_La_terre._clean.txt,"Jean, ce matin-là, un semoir de toile bleue no...",163687
5,1885_13_Germinal._clean.txt,"Dans la plaine rase, sous la nuit sans étoiles...",164473
6,1898_3_Paris._clean.txt,"Ce matin-là, vers la fin de janvier, l’abbé Pi...",177389
7,1901_2_Travail._clean.txt,"Dans sa promenade au hasard, Luc Froment, en s...",198712
8,1867_Therese_Raquin._clean.txt,"Au bout de la rue Guénégaud, lorsqu’on vient d...",66363
9,1883_11_Au_Bonheur_des_dames._clean.txt,Denise était venue à pied de la gare Saint-Laz...,147521


# Segmentation des Romans 

In [4]:
def segmenter_en_paquets(texte, taille_paquet=40): # Fonction pour segmenter le texte en paquets de phrases de taille spécifiée ici 40
    phrases = texte.splitlines()
    phrases = [p.strip() for p in phrases if p.strip()]
    
    paquets = []
    for i in range(0, len(phrases), taille_paquet):
        paquet = " ".join(phrases[i:i+taille_paquet])
        paquets.append(paquet)
        
    return paquets

rows = []

for _, row in df.iterrows(): # On itère sur chaque ligne du DataFrame
    paquets = segmenter_en_paquets(row["texte_brut"]) # On segmente le texte brut en paquets de phrases
    
    for idx, paquet in enumerate(paquets): # On itère sur chaque paquet de phrases
        rows.append({ # On ajoute une nouvelle ligne au tableau de bord pour chaque paquet de phrases
            "nom_fichier": row["nom_fichier"],
            "id_paquet": idx,
            "phrases_paquet": paquet
        })

df_phrases = pd.DataFrame(rows)

In [5]:
df_phrases.shape

(5327, 3)

In [6]:
chemin_sortie = Path("..") /"data" /"2_processed" /"01_paquets_phrases.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df_phrases.to_csv(chemin_sortie, index=False, encoding="utf-8")